# 03b - Build DWH (dw_ram_products)

Legge le 4 collection di staging da MongoDB Atlas (`st_ram_ddr4_it`, `st_ram_ddr5_it`, `st_ram_ddr4_de`, `st_ram_ddr5_de`), aggiunge i campi discriminanti `market` e `ram_category`, unisce tutto in un unico DataFrame e carica il risultato nella collection consolidata `dw_ram_products` (approccio flat, DB `idealo_ram`).

In [1]:
import os
import math
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = "idealo_ram"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

print("Collections presenti nel DB:", db.list_collection_names())

Collections presenti nel DB: ['dw_ram_products', 'st_ram_ddr5_de', 'sc_ram_ddr4_it', 'st_ram_ddr4_de', 'st_ram_ddr4_it', 'sc_ram_ddr4_de', 'sc_ram_ddr5_it', 'sc_ram_ddr5_de', 'st_ram_ddr5_it']


## 1. Lettura delle 4 collection di staging

In [2]:
# Mapping collection di staging -> (market, ram_category)
STAGING_SOURCES = {
    "st_ram_ddr4_it": ("IT", "DDR4"),
    "st_ram_ddr5_it": ("IT", "DDR5"),
    "st_ram_ddr4_de": ("DE", "DDR4"),
    "st_ram_ddr5_de": ("DE", "DDR5"),
}

dataframes = {}

for coll_name, (market, ram_category) in STAGING_SOURCES.items():
    docs = list(db[coll_name].find({}, {"_id": 0, 
                                        "characteristics_raw": 0, 
                                        "mainProductId": 0, "href": 0, 
                                        "hasProsAndCons": 0}))  # escludo _id di staging: nel DWH ne verrà generato uno nuovo
    df = pd.DataFrame(docs)
    df["market"] = market
    df["ram_category"] = ram_category
    dataframes[coll_name] = df
    print(f"{coll_name}: {len(df)} prodotti letti")

st_ram_ddr4_it: 174 prodotti letti
st_ram_ddr5_it: 152 prodotti letti
st_ram_ddr4_de: 174 prodotti letti
st_ram_ddr5_de: 156 prodotti letti


## 2. Controllo coerenza colonne tra i 4 set

Prima di concatenare, verifico che le 4 collection abbiano lo stesso schema di campi (a parte eventuali differenze attese).

In [3]:
col_sets = {name: set(df.columns) for name, df in dataframes.items()}
all_cols = set.union(*col_sets.values())

for name, cols in col_sets.items():
    missing = all_cols - cols
    if missing:
        print(f"{name}: mancano le colonne {missing}")
    else:
        print(f"{name}: OK, nessuna colonna mancante")

st_ram_ddr4_it: OK, nessuna colonna mancante
st_ram_ddr5_it: OK, nessuna colonna mancante
st_ram_ddr4_de: OK, nessuna colonna mancante
st_ram_ddr5_de: OK, nessuna colonna mancante


Se emergono colonne mancanti in qualche set, decidere qui come trattarle (es. fillna) prima di procedere con la concatenazione.

## 3. Concatenazione in un unico DataFrame

In [4]:
dw_df = pd.concat(dataframes.values(), ignore_index=True, sort=False)

print(f"Totale prodotti nel DWH: {len(dw_df)}")
print("\nDistribuzione per market/ram_category:")
print(dw_df.groupby(["market", "ram_category"]).size())

Totale prodotti nel DWH: 656

Distribuzione per market/ram_category:
market  ram_category
DE      DDR4            174
        DDR5            156
IT      DDR4            174
        DDR5            152
dtype: int64


## 4. Sanitize + caricamento su MongoDB

In [5]:
def sanitize_nans(records):
    """Sostituisce ogni NaN float (introdotto da pandas nei valori mancanti) con None vero."""
    for r in records:
        for k, v in r.items():
            if isinstance(v, float) and math.isnan(v):
                r[k] = None
    return records

records = dw_df.to_dict(orient="records")
records = sanitize_nans(records)

print(f"Record pronti per l'inserimento: {len(records)}")

Record pronti per l'inserimento: 656


In [6]:
DWH_COLLECTION = "dw_ram_products"

# Se il notebook viene rieseguito, ripulisco la collection per evitare duplicati
existing_count = db[DWH_COLLECTION].count_documents({})
if existing_count > 0:
    print(f"Trovati {existing_count} documenti gia' presenti in {DWH_COLLECTION}: li elimino prima di ricaricare.")
    db[DWH_COLLECTION].delete_many({})

result = db[DWH_COLLECTION].insert_many(records)
print(f"Inseriti {len(result.inserted_ids)} documenti in '{DWH_COLLECTION}'")

Trovati 656 documenti gia' presenti in dw_ram_products: li elimino prima di ricaricare.
Inseriti 656 documenti in 'dw_ram_products'


## 5. Verifica finale

In [7]:
total_in_db = db[DWH_COLLECTION].count_documents({})
print(f"Totale documenti in '{DWH_COLLECTION}': {total_in_db}")

print("\nSanity check - distribuzione per market/ram_category (letta direttamente da MongoDB):")
pipeline = [
    {"$group": {"_id": {"market": "$market", "ram_category": "$ram_category"}, "count": {"$sum": 1}}},
    {"$sort": {"_id.market": 1, "_id.ram_category": 1}},
]
for doc in db[DWH_COLLECTION].aggregate(pipeline):
    print(doc)

Totale documenti in 'dw_ram_products': 656

Sanity check - distribuzione per market/ram_category (letta direttamente da MongoDB):
{'_id': {'market': 'DE', 'ram_category': 'DDR4'}, 'count': 174}
{'_id': {'market': 'DE', 'ram_category': 'DDR5'}, 'count': 156}
{'_id': {'market': 'IT', 'ram_category': 'DDR4'}, 'count': 174}
{'_id': {'market': 'IT', 'ram_category': 'DDR5'}, 'count': 152}
